# test_nodes

Notebook tests for agent-loop routing and node execution.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'src' / 'task_router_graph').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate project root')

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from task_router_graph.nodes import route_node, execute_node, update_node
from task_router_graph.schema import Environment, Task


In [ ]:
class FakeLLM:
    def __init__(self, outputs):
        self.outputs = outputs
        self.i = 0

    def invoke(self, messages):
        if self.i >= len(self.outputs):
            raise RuntimeError('FakeLLM outputs exhausted')
        out = self.outputs[self.i]
        self.i += 1
        return SimpleNamespace(content=out)

env = Environment()
run_root = PROJECT_ROOT / 'var' / 'runs'
run_root.mkdir(parents=True, exist_ok=True)

controller_outputs = [
    '{"action_kind":"observe","tool":"ls","args":{"path":"."},"reason":"????"}',
    '{"action_kind":"generate_task","task_type":"normal","task_content":"?????????","reason":"????"}'
]
task, trace = route_node(
    llm=FakeLLM(controller_outputs),
    controller_system='[USER_INPUT]\n{{USER_INPUT}}\n[/USER_INPUT]\n[ROUNDS_JSON]\n{{ROUNDS_JSON}}\n[/ROUNDS_JSON]\n[SKILLS_INDEX]\n{{SKILLS_INDEX}}\n[/SKILLS_INDEX]',
    controller_skills_index='controller skills index',
    environment=env,
    user_input='??????????',
    workspace_root=PROJECT_ROOT,
    run_root=run_root,
    max_steps=3,
)
assert task.type == 'normal'
assert len(trace) == 2
assert trace[0].action_kind == 'observe'
assert trace[-1].action_kind == 'generate_task'
task, trace[-1].task_content


In [ ]:
normal_outputs = [
    '{"reply":"??????? code ????????","task_status":"done","task_result":"???????????"}'
]
task2, reply = execute_node(
    llm=FakeLLM(normal_outputs),
    normal_system='[TASK_CONTENT]\n{{TASK_CONTENT}}\n[/TASK_CONTENT]\n[ROUNDS_JSON]\n{{ROUNDS_JSON}}\n[/ROUNDS_JSON]\n[NORMAL_SKILLS_INDEX]\n{{NORMAL_SKILLS_INDEX}}\n[/NORMAL_SKILLS_INDEX]',
    normal_skills_index='normal skills index',
    environment=env,
    task=Task(type='normal', content='?????????'),
)
assert task2.status == 'done'
assert '??' in reply

env = update_node(env, '??????????', trace, task2, reply)
assert len(env.rounds) == 1
assert len(env.rounds[0].controller_trace) == 2
env.rounds[0].reply
